In [1]:
import pandas as pd
import numpy as np
import os


In [2]:
list_files = os.listdir("../")
list_files

['.git',
 'away_team.csv',
 'away_team_score.csv',
 'event.csv',
 'home_team.csv',
 'home_team_score.csv',
 'Javadi',
 'notebook.ipynb',
 'odds.csv',
 'pbp.csv',
 'power.csv',
 'round.csv',
 'season.csv',
 'statistics.csv',
 'time.csv',
 'tournament.csv',
 'venue.csv',
 'votes.csv']

In [3]:
match_time_df = pd.read_csv("../time.csv")

In [4]:
MIN_SET_SECONDS = 10 * 60
MAX_SET_SECONDS = 150 * 60
MIN_MATCH_SECONDS = 30 * 60
MAX_MATCH_SECONDS = 420 * 60
period_columns = ['period_1', 'period_2', 'period_3', 'period_4', 'period_5']
df = match_time_df.copy()
total_original = len(df)

In [5]:
df = df.drop_duplicates()

# STEP 2: Handle duplicate match_ids — distinguish corrupt vs interrupted

single_rows = df[~df['match_id'].duplicated(keep=False)].copy()
duplicate_ids = df[df['match_id'].duplicated(keep=False)].copy()

fixed_interrupted = [] 
corrupt_match_ids = [] 

for match_id, group in duplicate_ids.groupby('match_id'):

    # ── Test 1: Are the rows (nearly) identical? → corrupt ───────────────────
    numeric_columns = [c for c in period_columns if c in group.columns]
    period_values = group[numeric_columns].fillna(-1)
    is_identical = (period_values.nunique() == 1).all()
    if is_identical:
        corrupt_match_ids.append(match_id)
        continue

    # Test 2: Do any two rows share the same non-null set in the same slot? 
    #    e.g. both rows have a value for period_1 → overlapping, not interrupted
    conflict = False
    for col in period_columns:
        non_null_vals = group[col].dropna()
        if len(non_null_vals) > 1:
            conflict = True
            break
    if conflict:
        corrupt_match_ids.append(match_id)
        continue

    # Test 3: Timestamps must differ (sessions happened at different times) ─
    if 'current_period_start_timestamp' in group.columns:
        timestamps = group['current_period_start_timestamp'].dropna().unique()
        if len(timestamps) <= 1:
            # Only one unique timestamp across all rows → likely corrupt
            corrupt_match_ids.append(match_id)
            continue

    # Passes all tests: treat as interrupted match ──────────────────────────
    # Merge rows by taking the first non-null value per set column
    merged = {}
    merged['match_id'] = match_id
    for col in period_columns:
        non_null = group[col].dropna()
        merged[col] = non_null.iloc[0] if len(non_null) > 0 else np.nan

    # Keep the earliest timestamp as match start reference
    if 'current_period_start_timestamp' in group.columns:
        merged['current_period_start_timestamp'] = (
            group['current_period_start_timestamp'].dropna().min())

    fixed_interrupted.append(merged)

# Combine clean singles + resolved interrupted matches
interrupted_df = pd.DataFrame(fixed_interrupted) if fixed_interrupted else pd.DataFrame()
df = pd.concat([single_rows, interrupted_df], ignore_index=True)

n_corrupt = len(corrupt_match_ids)
n_interrupted = len(fixed_interrupted)


In [6]:
# STEP 3: Null out invalid individual set values

cells_nulled_s3 = 0
for col in period_columns:
    invalid_mask = df[col].notna() & (
        (df[col] < MIN_SET_SECONDS) | (df[col] > MAX_SET_SECONDS)
    )
    cells_nulled_s3 += invalid_mask.sum()
    df.loc[invalid_mask, col] = np.nan

In [7]:
# STEP 4: Remove matches with fewer than 2 valid sets

before_s4 = len(df)
df = df[df['period_1'].notna() & df['period_2'].notna()]
removed_s4 = before_s4 - len(df)


In [8]:
# STEP 5: Null out illogical set ordering

cells_nulled_s5 = 0
mask_no_p3 = df['period_3'].isna()
cells_nulled_s5 += (mask_no_p3 & df['period_4'].notna()).sum()
cells_nulled_s5 += (mask_no_p3 & df['period_5'].notna()).sum()
df.loc[mask_no_p3, ['period_4', 'period_5']] = np.nan
mask_no_p4 = df['period_4'].isna()
cells_nulled_s5 += (mask_no_p4 & df['period_5'].notna()).sum()
df.loc[mask_no_p4, 'period_5'] = np.nan


In [9]:
# STEP 6: Filter implausible total match durations

df['total_seconds'] = df[period_columns].fillna(0).sum(axis=1)
df['total_minutes'] = (df['total_seconds'] / 60).round(1)
df['total_hours']   = (df['total_seconds'] / 3600).round(2)

before_s6 = len(df)
df = df[
    (df['total_seconds'] >= MIN_MATCH_SECONDS) &
    (df['total_seconds'] <= MAX_MATCH_SECONDS)
]
removed_s6 = before_s6 - len(df)


In [10]:
total_clean   = len(df)
total_removed = total_original - total_clean
pct_clean     = round((total_clean   / total_original) * 100, 2)
pct_unclean   = round((total_removed / total_original) * 100, 2)

print("\n DATA CLEANING REPORT (with interruption handling)\n")
print(f"  Original dataset size          : {total_original:,} rows")
print(f"  Corrupt duplicates removed     : {n_corrupt:,} match groups")
print(f"  Interrupted matches resolved   : {n_interrupted:,} matches merged")
print(f"  Rows after cleaning            : {total_clean:,} rows")



 DATA CLEANING REPORT (with interruption handling)

  Original dataset size          : 35,671 rows
  Corrupt duplicates removed     : 1,848 match groups
  Interrupted matches resolved   : 91 matches merged
  Rows after cleaning            : 9,069 rows


In [11]:
# Top 10 longest matches
top_10 = (df[['match_id', 'total_seconds', 'total_minutes', 'total_hours']]
            .sort_values('total_seconds', ascending=False)
            .head(10))
print("\nTOP 10 LONGEST MATCHES:\n")
print(top_10.to_string(index=False))

longest = df.loc[df['total_seconds'].idxmax()]
print(f"\nLongest match — ID : {longest['match_id']}")
print(f"Duration           : {longest['total_minutes']} min ({longest['total_hours']} hrs)")


TOP 10 LONGEST MATCHES:

 match_id  total_seconds  total_minutes  total_hours
 12160369        18123.0          302.0         5.03
 12056393        17834.0          297.2         4.95
 12174579        16359.0          272.6         4.54
 12201395        15929.0          265.5         4.42
 12152568        15649.0          260.8         4.35
 12163362        15505.0          258.4         4.31
 12194966        15240.0          254.0         4.23
 12063596        14922.0          248.7         4.14
 12172227        14766.0          246.1         4.10
 12159529        14737.0          245.6         4.09

Longest match — ID : 12160369.0
Duration           : 302.0 min (5.03 hrs)
